# 🧪 Test Langfuse Prompt Management (`supervisor` Prompt)

Notebook này dùng để:
1. Kết nối đến Langfuse Server (`http://localhost:3000`)
2. Lấy Prompt **`supervisor`** (phiên bản `production` / `latest`) từ Langfuse
3. Compile Prompt với biến đầu vào (`query`, `chat_history`)
4. Chạy phân loại Intent thực tế qua LLM và tự động liên kết Trace vào **Linked Generations** trên Langfuse

In [1]:
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# 1. Load biến môi trường từ file .env ở thư mục gốc dev_llm_service
env_path = Path.cwd().parent / ".env"
if not env_path.exists():
    env_path = Path.cwd() / ".env"
load_dotenv(dotenv_path=env_path, override=True)

# 2. Thêm thư mục gốc vào sys.path để import được app module
workspace_dir = str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

from app.core.config import settings
from app.llmops.langfuse import get_langfuse_client

# 3. Khởi tạo Langfuse Client
langfuse = get_langfuse_client()

print(f"✅ Langfuse Host      : {getattr(settings, 'LANGFUSE_HOST', 'http://localhost:3000')}")
print(f"✅ Langfuse Public Key: {getattr(settings, 'LANGFUSE_PUBLIC_KEY', '')[:12]}...")
print(f"✅ Client Ready       : {langfuse is not None}")

# Kiểm tra xác thực (auth check)
if langfuse:
    try:
        is_ok = langfuse.auth_check()
        print(f"🔐 Auth Check Status : {'SUCCESS' if is_ok else 'FAILED'}")
    except Exception as e:
        print(f"⚠️ Auth check warning: {e}")

c:\2_Company\GSOFT\Enterprice-Chatbot\BVBank-Chatbot\dev_llm_service\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Langfuse Host      : http://localhost:3000
✅ Langfuse Public Key: pk-lf-096410...
✅ Client Ready       : True
🔐 Auth Check Status : SUCCESS


## 📥 Bước 1: Fetch Prompt `supervisor` từ Langfuse

In [2]:
# Lấy prompt 'supervisor' từ Langfuse
try:
    prompt = langfuse.get_prompt("supervisor")
    print(f"📌 Prompt Name    : {prompt.name}")
    print(f"📌 Version        : {prompt.version}")
    print(f"📌 Client Class   : {type(prompt).__name__}")
    print(f"📌 Labels         : {getattr(prompt, 'labels', [])}")
    print(f"📌 Config         : {getattr(prompt, 'config', {})}")
    print("-" * 50)
    
    # In nội dung prompt
    if isinstance(prompt.prompt, list):
        print(f"💬 Chat Messages in Prompt ({len(prompt.prompt)} message(s)):")
        for idx, msg in enumerate(prompt.prompt):
            role = msg.get('role', 'unknown')
            content = msg.get('content', '')
            print(f"  [{role.upper()}]:\n{content[:300]}...\n")
    else:
        print(f"📄 Text Prompt Content:\n{str(prompt.prompt)[:300]}...")
        
except Exception as e:
    print(f"❌ Không thể tải prompt: {e}")

📌 Prompt Name    : supervisor
📌 Version        : 1
📌 Client Class   : ChatPromptClient
📌 Labels         : ['production', 'latest']
📌 Config         : {'temperature': 0}
--------------------------------------------------
💬 Chat Messages in Prompt (2 message(s)):
  [SYSTEM]:
You are an expert Intent Classifier for the BVBank AI Assistant system. 
Your sole task is to analyze the user's query and classify it into exactly ONE of the four intent categories (`faq`, `rag`, `procurement`, or `fallback`).

### 1. INTENT CATEGORY DEFINITIONS

1. `faq` (Frequently Asked Question...

  [USER]:
### FEW-SHOT EXAMPLES FOR REFERENCE
[
    {
        "query": "Khi gặp sự cố CNTT, tôi liên hệ hỗ trợ ở đâu?",
        "output": {
            "reasoning": "Câu hỏi tra cứu một bước, đáp án cố định về kênh hỗ trợ CNTT.",
            "intent": "faq",
            "query": "Liên hệ hỗ trợ sự cố CNTT",
 ...



## ⚙️ Bước 2: Compile Prompt với Dữ Liệu Đầu Vào (Variables)

In [3]:
test_query = "Quy trình mua sắm bàn ghế cho phòng giao dịch mới cần những tờ trình gì?"
test_history = ""

# Compile prompt với các biến đầu vào
try:
    compiled_prompt = prompt.compile(
        query=test_query,
        chat_history=test_history,
    )
    print("✅ Compiled Prompt thành công!")
    print(f"Kiểu dữ liệu compiled: {type(compiled_prompt)}\n")
    
    if isinstance(compiled_prompt, list):
        for msg in compiled_prompt:
            print(f"[{msg.get('role', '').upper()}]:")
            print(msg.get('content', '')[:300] + "...\n")
    else:
        print(str(compiled_prompt)[:500] + "...")
except Exception as e:
    print(f"⚠️ Lưu ý khi compile: {e}")
    compiled_prompt = prompt.compile()
    print("✅ Compiled fallback thành công!")

✅ Compiled Prompt thành công!
Kiểu dữ liệu compiled: <class 'list'>

[SYSTEM]:
You are an expert Intent Classifier for the BVBank AI Assistant system. 
Your sole task is to analyze the user's query and classify it into exactly ONE of the four intent categories (`faq`, `rag`, `procurement`, or `fallback`).

### 1. INTENT CATEGORY DEFINITIONS

1. `faq` (Frequently Asked Question...

[USER]:
### FEW-SHOT EXAMPLES FOR REFERENCE
[
    {
        "query": "Khi gặp sự cố CNTT, tôi liên hệ hỗ trợ ở đâu?",
        "output": {
            "reasoning": "Câu hỏi tra cứu một bước, đáp án cố định về kênh hỗ trợ CNTT.",
            "intent": "faq",
            "query": "Liên hệ hỗ trợ sự cố CNTT",
 ...



## 🚀 Bước 3: Chạy Test Phân Loại Ý Định (Intent Classification) với LLM & Ghi Trace lên Langfuse

In [4]:
import asyncio
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from app.llmops.factory import get_chat_model
from app.ai.agent.supervisor.schemas import RouterOutput
from app.llmops.langfuse import get_langfuse_callback, flush_langfuse

# 1. Khởi tạo Chat Model và ép kiểu Structured Output (RouterOutput)
llm = get_chat_model()
structured_llm = llm.with_structured_output(RouterOutput)

# 2. Danh sách các câu hỏi test mẫu cho các Intent
test_cases = [
    "Quy trình mua sắm bàn ghế cho phòng giao dịch mới cần những tờ trình gì?",
    "Hướng dẫn cách tạo tờ trình mua sắm trên phần mềm gAMSPro",
    "Thời gian làm việc của ngân hàng BVBank là mấy giờ?",
    "Xin chào bạn ơi, chúc ngày mới tốt lành"
]

print(f"🤖 Model: {getattr(settings, 'LLM_MODEL', 'default')} (Provider: {getattr(settings, 'AI_PROVIDER', 'llm')})\n")

for idx, query in enumerate(test_cases, 1):
    # Callback gán prompt để liên kết generation vào Prompt trên Langfuse
    cb = get_langfuse_callback(
        session_id="test-session-notebook-8",
        user_id="tester-01",
        tags=["notebook-test", "supervisor-prompt"],
        trace_name=f"Test-Supervisor: Case #{idx}",
    )
    
    # Chuẩn bị messages từ Langfuse prompt
    try:
        compiled = prompt.compile(query=query, chat_history="")
    except Exception:
        compiled = prompt.compile()
        
    langchain_messages = []
    if isinstance(compiled, list):
        for m in compiled:
            role = m.get("role", "user")
            content = m.get("content", "")
            if role == "system":
                langchain_messages.append(SystemMessage(content=content))
            elif role == "user":
                langchain_messages.append(HumanMessage(content=content))
            elif role in ("assistant", "ai"):
                langchain_messages.append(AIMessage(content=content))
        
        # Nếu prompt trên Langfuse chỉ có System message, bổ sung Human message chứa query
        if len(langchain_messages) == 1 and isinstance(langchain_messages[0], SystemMessage):
            langchain_messages.append(HumanMessage(content=query))
    else:
        langchain_messages = [
            SystemMessage(content=str(compiled)),
            HumanMessage(content=query)
        ]
    
    # Gọi LLM (Async invoke)
    config = {"callbacks": [cb]} if cb else {}
    result: RouterOutput = await structured_llm.ainvoke(langchain_messages, config=config)
    
    print(f"Case #{idx}: '{query}'")
    print(f"  👉 Intent     : {result.intent.upper()}")
    print(f"  👉 Confidence : {result.confidence:.2f}")
    print(f"  👉 Reasoning  : {result.reasoning}")
    print("-" * 60)

# Đẩy toàn bộ trace lên Langfuse
flush_langfuse()
print("\n🎉 Đã hoàn tất và gửi toàn bộ Traces lên Langfuse! Hãy vào tab 'Linked Generations' trên Langfuse để xem.")

🤖 Model: qwen3:1.7b (Provider: vllm)

Case #1: 'Quy trình mua sắm bàn ghế cho phòng giao dịch mới cần những tờ trình gì?'
  👉 Intent     : PROCUREMENT
  👉 Confidence : 0.95
  👉 Reasoning  : Câu hỏi liên quan đến quy trình mua sắm bàn ghế trong hệ thống gAMSPro, thuộc phạm vi nghiệp vụ procurement. Không có đại từ quy chiếu đến đối tượng cụ thể, nhưng yêu cầu tra cứu thông tin về tờ trình mua sắm là hoạt động chuẩn xác trong phân hệ procurement.
------------------------------------------------------------
Case #2: 'Hướng dẫn cách tạo tờ trình mua sắm trên phần mềm gAMSPro'
  👉 Intent     : PROCUREMENT
  👉 Confidence : 0.95
  👉 Reasoning  : Câu hỏi hướng dẫn tạo tờ trình mua sắm thuộc phạm vi quy trình thao tác phê duyệt và tạo tờ trình trên phần mềm gAMSPro, cần tra cứu tài liệu hướng dẫn sử dụng phần mềm.
------------------------------------------------------------
Case #3: 'Thời gian làm việc của ngân hàng BVBank là mấy giờ?'
  👉 Intent     : FAQ
  👉 Confidence : 0.95
  👉 Reasoning  :